### Conversational Chat bot demo

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv


In [3]:
load_dotenv()
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
)

In [3]:
llm_response = llm.invoke("Tell me a Joke!")
print(llm_response)

content='Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 12, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXf6soMXsRvr6pIYAISQOo8Mktsts', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--9be374bf-256d-4c53-8a5b-42d8ed8afa42-0' usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### Parsing output

In [4]:
parser = StrOutputParser()
parser.invoke(llm_response)

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

### Simple chain

In [5]:
simple_chain = llm | parser
response = simple_chain.invoke("Tell me a Joke!")
print(response)

Why did the scarecrow win an award?

Because he was outstanding in his field!


### Structured Output

In [6]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Rating from 1 to 5")
    pros: List[str] = Field(description="Positive aspects of the phone")
    cons: List[str] = Field(description="Negative aspects of the phone")
    review_summary: str = Field(description="Summary of the review")

review_text = """
Just got my hands on google pixel fold, it's a beast! the screen is gorgeous, colors pops like crazy.
I've been using it for a few days now and I'm loving it. I can't wait to get home and show it off to my family.
The camera quality is outstanding, even in low light conditions. It captures everything perfectly.
But I must say that the battery life is a bit of a concern. It's not as long as I expected, but that's not a big deal.
Overall, I'm really impressed with this phone it is solid 4 star rating. It's a great value for the money and I highly recommend it.
"""

structured_llm = llm.with_structured_output(MobileReview)

structured_response = structured_llm.invoke(review_text)
print(structured_response)

phone_model='Google Pixel Fold' rating=4.0 pros=['Gorgeous screen with vibrant colors', 'Outstanding camera quality, especially in low light', 'Great value for the money'] cons=['Battery life could be better'] review_summary='Overall, the Google Pixel Fold is an impressive device with a stunning display and excellent camera performance, though the battery life is a slight drawback.'


### Prompt Template

- Helps create dynamic prompts

In [7]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    "Tell me a short story about {topic}"
)

prompt.invoke("Space")

ChatPromptValue(messages=[HumanMessage(content='Tell me a short story about Space', additional_kwargs={}, response_metadata={})])

In [8]:
chain_with_prompt = prompt | llm | parser
response = chain_with_prompt.invoke("Cars")
print(response)

Once upon a time in the bustling town of Autoville, cars weren’t just machines; they were vibrant characters with personalities. Each car had its own quirks and dreams. Among them was a little blue hatchback named Benny. Benny was small and often overlooked, but he had a big heart and an even bigger dream: to race in the annual Autoville Grand Prix.

Every year, the Grand Prix attracted the fastest and flashiest cars, like the sleek red sports car, Blaze, and the powerful black muscle car, Titan. Benny admired them from afar, wishing he could join the race. His friends, a wise old van named Vinnie and a cheerful electric car named Zoe, encouraged him to believe in himself.

“Benny, it’s not about size or speed; it’s about heart and determination,” Vinnie said, his headlights twinkling with wisdom.

With newfound confidence, Benny decided to enter the race. He spent weeks preparing, tuning his engine and practicing on the winding roads of Autoville. The day of the Grand Prix arrived, an

### Different LLM Messages

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_message = SystemMessage(
    content="You are a helpful assistant that translates English to French."
)
human_message = HumanMessage(content="I love programming.")

load_dotenv()
parser = StrOutputParser()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)


response = llm.invoke([system_message, human_message])
print(response)




content="J'aime la programmation." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 26, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXgd4wL2TALWC7ifKT9nUi7RmPtjm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--66440432-6b23-4ba0-b990-ccf29c311a39-0' usage_metadata={'input_tokens': 26, 'output_tokens': 5, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [11]:
prompt = ChatPromptTemplate(
    [
        ("system", "You are an helpful assistant that tell amazing jokes."),
        ("human", "Tell me a joke about {topic}."),
    ]
)

prompt_value = prompt.invoke({"topic": "Space"})
print(prompt_value)

response = llm.invoke(prompt_value)
print(response)

messages=[SystemMessage(content='You are an helpful assistant that tell amazing jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about Space.', additional_kwargs={}, response_metadata={})]
content='Why did the sun go to school?\n\nTo get a little brighter!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 28, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CXgjJVGf9hff8s4uQpvz5lO6kiZqB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--34125f19-1eb8-4a3b-9007-566bde5676fd-0' usage_metadata={'input_tokens': 28, 'output_tokens': 14, 'total_tokens': 42, 'input_token_details': {'audio': 0